# Entropy of a Categorical Distribution

Chapter 14: Information Theory · *Mathematics for AI and Machine Learning* by Fuheng Wu

Simplex heatmap, probability partition, and −p_i log p_i decomposition for 3-class entropy.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-14/entropy-categorical/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's default probabilities (third class takes the rest)
p1, p2 = 0.33, 0.33
p = torch.tensor([p1, p2, 1 - p1 - p2], dtype=torch.float64)


def entropy(p):
    """H(p) = -sum p_i log p_i in nats (0 log 0 = 0)."""
    p = p[p > 0]
    return -(p * p.log()).sum()


print("p =", p.tolist())
print("contributions -p_i log p_i =", (-(p * p.log())).tolist())
print(f"H(p) = {entropy(p).item():.6f} nats = {entropy(p).item() / math.log(2):.6f} bits")
print("torch Categorical entropy =", torch.distributions.Categorical(probs=p).entropy().item())

# ── The demo's presets: uniform is the maximum log 3 ────────────────────────
for name, q in {"uniform": [1 / 3, 1 / 3, 1 / 3], "skewed": [0.6, 0.3, 0.1], "certain": [0.92, 0.05, 0.03]}.items():
    q = torch.tensor(q, dtype=torch.float64)
    print(f"{name:8s} {[round(v, 2) for v in q.tolist()]}: H = {entropy(q).item():.4f}  (max log 3 = {math.log(3):.4f})")

# ── Search the simplex: no distribution beats the uniform one ───────────────
torch.manual_seed(0)
samples = torch.distributions.Dirichlet(torch.ones(3)).sample((200000,)).double()
H = -(samples * samples.clamp_min(1e-300).log()).sum(1)
print("\nmax entropy over 200000 random distributions =", H.max().item(), "at", samples[H.argmax()].tolist())

# Entropy = expected surprise: average of -log p(x) over samples x ~ p
x = torch.distributions.Categorical(probs=p).sample((200000,))
print("Monte Carlo E[-log p(X)] =", (-p[x].log()).mean().item())

p = [0.33, 0.33, 0.3399999999999999]
contributions -p_i log p_i = [0.3658586660921317, 0.3658586660921317, 0.3667952848664562]
H(p) = 1.098513 nats = 1.584819 bits
torch Categorical entropy = 1.0985126170507196
uniform  [0.33, 0.33, 0.33]: H = 1.0986  (max log 3 = 1.0986)
skewed   [0.6, 0.3, 0.1]: H = 0.8979  (max log 3 = 1.0986)
certain  [0.92, 0.05, 0.03]: H = 0.3317  (max log 3 = 1.0986)

max entropy over 200000 random distributions = 1.0986095881299833 at [0.3323096036911011, 0.3335076570510864, 0.3341827392578125]
Monte Carlo E[-log p(X)] = 1.0985085869006943
